In [4]:
import sys
from pathlib import Path 

repo_root = Path.cwd().parents[1]
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

In [6]:
import numpy as np
import pandas as pd
import time
import plotly.express as px
import plotly.graph_objects as go
import os
import plotly.io as pio
pio.templates.default = "plotly_white"

from pathlib import Path
from tqdm.autonotebook import tqdm
import warnings
import humanize

#If importing darts is throwing an error, import torch beforehand and then import darts
# import torch
from darts import TimeSeries
from darts.models import (
    NaiveSeasonal,
    NaiveMean,
    NaiveDrift,
    ExponentialSmoothing,
    AutoARIMA,
    ARIMA,
    Theta,
    FFT
)

from darts.metrics import mase, mse, mae, ope

from src.utils.ts_utils import forecast_bias
from src.utils.general import LogTime
from src.utils import plotting_utils
# %load_ext autoreload
# %autoreload 2
np.random.seed(42)
tqdm.pandas()

In [9]:
os.makedirs("imgs", exist_ok=True)
preprocessed = Path('/Users/Soham/Documents/FALL2026/MTFS_textbook_work/data/london_smart_meters/preprocessed')

In [ ]:
def format_plot(fig, legends = None, xlabel="Time", ylabel="Value", title="", font_size=15):
    if legends:
        names = cycle(legends)
        fig.for_each_trace(lambda t:  t.update(name = next(names)))
    fig.update_layout(
            autosize=False,
            width=900,
            height=500,
            title_text=title,
            title={
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top'},
            titlefont={
                "size": 20
            },
            legend_title = None,
            legend=dict(
                font=dict(size=font_size),
                orientation="h",
                yanchor="bottom",
                y=0.98,
                xanchor="right",
                x=1,
            ),
            yaxis=dict(
                title_text=ylabel,
                titlefont=dict(size=font_size),
                tickfont=dict(size=font_size),
            ),
            xaxis=dict(
                title_text=xlabel,
                titlefont=dict(size=font_size),
                tickfont=dict(size=font_size),
            )
        )
    return fig

In [10]:
try :
    train_df = pd.read_parquet(preprocessed / "selected_blocks_train_missing_imputed.parquet")
    val_df = pd.read_parquet(preprocessed / "selected_blocks_val_missing_imputed.parquet")
    test_df = pd.read_parquet(preprocessed / "selected_blocks_test_missing_imputed.parquet")
except FileNotFoundError:
    warnings.warn("The London Smart Meter dataset is not found. Please download it from https://data.london.gov.uk/dataset/smartmeter-energy-use-data-in-london-households and place it in the data/london_smart_meters directory.")

In [11]:
len(train_df.LCLid.unique())

300

In [12]:
freq = train_df.iloc[0]['frequency']
ts_train = train_df.loc[train_df.LCLid=="MAC000193", ["timestamp","energy_consumption"]].set_index("timestamp")
ts_val = val_df.loc[val_df.LCLid=="MAC000193", ["timestamp","energy_consumption"]].set_index("timestamp")
ts_test = test_df.loc[test_df.LCLid=="MAC000193", ["timestamp","energy_consumption"]].set_index("timestamp")